### ***Next Phase: Hyperparameter Tuning***

As we had planned, we'll tune only:

LightGBM (your strongest baseline)
XGBoost (typically competitive after tuning)

We'll leave Random Forest, Decision Tree, Logistic Regression, etc., as baseline comparisons.

### ***Using Optuna***
**Optuna** is currently one of the best hyperparameter optimization frameworks for tree-based models like LightGBM and XGBoost. Compared to Grid Search, it explores the search space intelligently using Bayesian optimization (specifically the Tree-structured Parzen Estimator, or TPE), so it usually finds better hyperparameters with fewer trials.

In [ ]:
# !pip install optuna

In [16]:
# import required libraries
import optuna
import xgboost as xgb
import lightgbm as lgb
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
PROJECT_ROOT = Path.cwd().parent      # if notebook is inside notebooks/
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(exist_ok=True)

### Importing the datasets

In [13]:
X_train = pd.read_csv(DATA_DIR / "processed/base_X_train.csv")
print(X_train.shape)
y_train = pd.read_csv(DATA_DIR / "processed/base_y_train.csv")
print(y_train.shape)
X_test = pd.read_csv(DATA_DIR / "processed/base_X_test.csv")
print(X_test.shape)
y_test = pd.read_csv(DATA_DIR / "processed/base_y_test.csv")
print(y_test.shape)

(40972, 56)
(40972, 1)
(10243, 56)
(10243, 1)


### ***Define the Objective Function***

We'll optimize Macro F1 Score using 5-fold Stratified Cross Validation.

In [9]:
def objective(trial):

    params = {
        "objective": "multiclass",          
        "metric": "multi_logloss",
        "num_class": len(np.unique(y_train)),   

        "boosting_type": "gbdt",

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.3, log=True
        ),

        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves", 20, 200
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 15
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples", 5, 100
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.5, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.5, 1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10.0, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-8, 10.0, log=True
        ),

        "random_state": 42,

        "n_jobs": -1
    }

    model = lgb.LGBMClassifier(**params)

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1
    ).mean()

    return score

### Run Optuna

In [10]:
study = optuna.create_study(
    direction="maximize",
    study_name="LightGBM Optimization"
)

study.optimize(
    objective,
    n_trials=50,
    show_progress_bar=True
)

[I 2026-07-12 20:56:03,647] A new study created in memory with name: LightGBM Optimization


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-07-12 20:56:35,254] Trial 0 finished with value: 0.6873205325453097 and parameters: {'learning_rate': 0.27009879790690433, 'n_estimators': 324, 'num_leaves': 65, 'max_depth': 12, 'min_child_samples': 100, 'subsample': 0.5804829748085937, 'colsample_bytree': 0.6087796616561678, 'reg_alpha': 1.8251010820999264e-05, 'reg_lambda': 1.1978410102473678}. Best is trial 0 with value: 0.6873205325453097.
[I 2026-07-12 20:56:42,002] Trial 1 finished with value: 0.6845383916497945 and parameters: {'learning_rate': 0.07685229804418846, 'n_estimators': 208, 'num_leaves': 20, 'max_depth': 3, 'min_child_samples': 84, 'subsample': 0.613487830920753, 'colsample_bytree': 0.5142156356441543, 'reg_alpha': 2.757252787321183e-06, 'reg_lambda': 2.5534231042664492e-05}. Best is trial 0 with value: 0.6873205325453097.
[I 2026-07-12 20:56:49,990] Trial 2 finished with value: 0.6960349264406899 and parameters: {'learning_rate': 0.10912105425211346, 'n_estimators': 125, 'num_leaves': 179, 'max_depth': 6, '

***Best Parameters***

In [11]:
print("Best F1 Macro:", study.best_value)
print()

print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"{key}: {value}")

Best F1 Macro: 0.7003113993507958

Best Parameters:
learning_rate: 0.014580609126384928
n_estimators: 556
num_leaves: 46
max_depth: 8
min_child_samples: 63
subsample: 0.6575749331360101
colsample_bytree: 0.9449241981913695
reg_alpha: 0.00036726665199474117
reg_lambda: 5.499811280879408e-07


### This is the best Macro F1 Score we could find by applying Hyperparameter tuning using Optuna

**Using the same Hyperparamters to Train the final lightgbm model**

In [12]:
best_params = study.best_params.copy()

best_params.update({
    "objective": "multiclass",
    "metric": "multi_logloss",
    "num_class": len(np.unique(y_train)),
    "random_state": 42,
    "n_jobs": -1
})

best_model = lgb.LGBMClassifier(**best_params)

best_model.fit(X_train, y_train)

c:\Users\Pradhuman\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\Pradhuman\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004234 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4070
[LightGBM] [Info] Number of data points in the train set: 40972, number of used features: 55
[LightGBM] [Info] Start training from score -2.182061
[LightGBM] [Info] Start training from score -0.466281
[LightGBM] [Info] Start training from score -1.929162
[LightGBM] [Info] Start training from score -2.166391
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

,boosting_type,'gbdt'
,num_leaves,46
,max_depth,8
,learning_rate,0.014580609126384928
,n_estimators,556
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,63


### Evaluating The Results

In [14]:
y_pred = best_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="macro"))
print("Recall   :", recall_score(y_test, y_pred, average="macro"))
print("F1 Score :", f1_score(y_test, y_pred, average="macro"))

Accuracy : 0.7951771941813922
Precision: 0.7183412581758801
Recall   : 0.6871171233609572
F1 Score : 0.697770658864942


In [15]:
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.79      0.80      1155
           1       0.84      0.92      0.88      6426
           2       0.46      0.30      0.36      1488
           3       0.76      0.74      0.75      1174

    accuracy                           0.80     10243
   macro avg       0.72      0.69      0.70     10243
weighted avg       0.77      0.80      0.78     10243

[[ 915  240    0    0]
 [ 177 5922  296   31]
 [  45  762  442  239]
 [   0   89  219  866]]


***"Why does Class 2 perform poorly?"***

"The confusion matrix shows that most Class 2 instances are misclassified as Class 1 or Class 3 rather than distant classes. This suggests the feature space for Class 2 overlaps with adjacent risk categories. Since these classes represent neighboring credit risk levels, their financial characteristics are naturally similar, making them harder to separate. The model performs well on the majority class and the extreme classes, while the intermediate class remains the most challenging."

### Hyperparameter tuning Of XGBoost

In [19]:
def objective(trial):

    params = {

        "objective": "multi:softprob",          
        "num_class": len(np.unique(y_train)),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            1000
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            15
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.5,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.5,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-8,
            10,
            log=True
        ),

        "random_state": 42,

        "n_jobs": -1,

        "eval_metric": "mlogloss",

        "tree_method": "hist"
    }

    model = xgb.XGBClassifier(**params)

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1
    ).mean()

    return score

**Create Study**

In [20]:
study = optuna.create_study(
    direction="maximize",
    study_name="XGBoost Optimization"
)

# start Optimization

study.optimize(
    objective,
    n_trials=50,
    show_progress_bar=True
)

[I 2026-07-12 21:51:27,527] A new study created in memory with name: XGBoost Optimization


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-07-12 21:51:43,257] Trial 0 finished with value: 0.6651679685367111 and parameters: {'learning_rate': 0.016644431148159975, 'n_estimators': 434, 'max_depth': 4, 'min_child_weight': 3, 'subsample': 0.5492268851081492, 'colsample_bytree': 0.5764840860905815, 'gamma': 2.0752567463719434, 'reg_alpha': 0.0006662076582143514, 'reg_lambda': 4.045067807742674e-05}. Best is trial 0 with value: 0.6651679685367111.
[I 2026-07-12 21:52:02,600] Trial 1 finished with value: 0.6998929327899417 and parameters: {'learning_rate': 0.10201761314307098, 'n_estimators': 454, 'max_depth': 14, 'min_child_weight': 8, 'subsample': 0.5678889406237941, 'colsample_bytree': 0.9770809054051204, 'gamma': 1.6729514662792226, 'reg_alpha': 0.014929270079190128, 'reg_lambda': 2.727809371249588e-07}. Best is trial 1 with value: 0.6998929327899417.
[I 2026-07-12 21:52:53,514] Trial 2 finished with value: 0.7012624027574519 and parameters: {'learning_rate': 0.011521631688992095, 'n_estimators': 992, 'max_depth': 10,

### Best HyperParameter

In [21]:
print("Best Macro F1:", study.best_value)

print("\nBest Parameters:")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

Best Macro F1: 0.7046930581813624

Best Parameters:
learning_rate: 0.10949772139283392
n_estimators: 719
max_depth: 12
min_child_weight: 5
subsample: 0.6781444065566522
colsample_bytree: 0.8854318824344669
gamma: 2.4253844786820955
reg_alpha: 0.010108092538742498
reg_lambda: 7.309709355503078


### Training Final XGBoost Model ON Best parameters

In [27]:
best_params = study.best_params.copy()

best_params.update({

    "objective": "multi:softprob",

    "num_class": len(np.unique(y_train)),

    "random_state": 42,

    "eval_metric": "mlogloss",

    "tree_method": "hist",

    "n_jobs": -1

})

final_xgb_model = xgb.XGBClassifier(**best_params)

final_xgb_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8854318824344669
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import l

In [30]:
y_pred = final_xgb_model.predict(X_test)

In [31]:
y_prob = final_xgb_model.predict_proba(X_test)

In [33]:
y_pred = final_xgb_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="macro"))
print("Recall   :", recall_score(y_test, y_pred, average="macro"))
print("F1 Score :", f1_score(y_test, y_pred, average="macro"))

Accuracy : 0.7930293859220932
Precision: 0.7126939226586432
Recall   : 0.6860273372296302
F1 Score : 0.6954834660316951


In [34]:
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.79      0.80      1155
           1       0.85      0.92      0.88      6426
           2       0.45      0.30      0.36      1488
           3       0.75      0.73      0.74      1174

    accuracy                           0.79     10243
   macro avg       0.71      0.69      0.70     10243
weighted avg       0.77      0.79      0.78     10243

[[ 914  241    0    0]
 [ 178 5899  316   33]
 [  45  740  451  252]
 [   0   75  240  859]]


In [32]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(
    y_test,
    y_prob,
    multi_class="ovr",
    average="macro"
)

print("ROC-AUC:", roc_auc)

ROC-AUC: 0.9342522051093778


In [35]:

joblib.dump(
    best_model,
    MODEL_DIR / "final_lgbm_model.pkl"
)
joblib.dump(
    final_xgb_model,
    MODEL_DIR / "final_xgb_model.pkl"
)

['d:\\Study\\IFRS-9-Complient-Risk-Analysis-modelling\\models\\final_xgb_model.pkl']